In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
ruta = "../data/processed/tabla_principal_ventas.csv"

In [4]:
df = pd.read_csv(ruta)

print("Tabla cargada correctamente.")
print(df.columns)

Tabla cargada correctamente.
Index(['item_id', 'order_id', 'user_id', 'product_id', 'inventory_item_id',
       'status', 'created_at', 'shipped_at', 'delivered_at', 'returned_at',
       'revenue', 'cost', 'category', 'brand', 'distribution_center_id',
       'dist_center_name', 'num_of_item', 'country', 'gender', 'age', 'margin',
       'margin_percentage', 'is_returned', 'mes', 'year', 'es_recurrente',
       'estacion'],
      dtype='object')


A. Finanzas y Rentabilidad 


En esta sección analizamos el rendimiento económico de la plataforma. El objetivo es identificar qué categorías y marcas generan el mayor Margen Total, diferenciándolas de aquellas que tienen un alto volumen de ventas pero baja rentabilidad unitaria. 
Para el cálculo del Margen Porcentual utilizamos la siguiente fórmula:$$Margen \% = \frac{\text{Margen Total}}{\text{Ventas Totales}} \times 100$$

In [8]:
finanzas = df.groupby(['category', 'brand']).agg({
    'revenue': 'sum',
    'margin': 'sum',
    'item_id': 'count',
    'is_returned': 'sum',
    'es_recurrente': 'sum'
}).reset_index()

In [9]:
finanzas.columns = ['Categoria', 'Marca', 'Ventas_Totales', 'Margen_Total', 'Unidades', 'Devoluciones', 'Ventas_Recurrentes']

In [10]:
finanzas['Rentabilidad (%)'] = (finanzas['Margen_Total'] / finanzas['Ventas_Totales']) * 100

In [11]:
finanzas['Tasa_Devolucion (%)'] = (finanzas['Devoluciones'] / finanzas['Unidades']) * 100

In [12]:
finanzas['Ticket_Medio'] = finanzas['Ventas_Totales'] / finanzas['Unidades']

In [13]:
finanzas['%_Recurrencia'] = (finanzas['Ventas_Recurrentes'] / finanzas['Unidades']) * 100

In [14]:
finanzas.to_csv("../data/processed/kpi_finanzas.csv", index=False)

In [15]:
finanzas.sort_values(by='Margen_Total', ascending=False).head()

,Categoria,Marca,Ventas_Totales,Margen_Total,Unidades,Devoluciones,Ventas_Recurrentes,Rentabilidad (%),Tasa_Devolucion (%),Ticket_Medio,%_Recurrencia
1805,Jeans,7 For All Mankind,155347.520485,72444.999511,979,95,768,46.634152,9.703779,158.679796,78.447395
1995,Jeans,True Religion,121238.010132,56278.829113,504,55,406,46.420119,10.912698,240.551607,80.555556
2441,Outerwear & Coats,Carhartt,99286.679039,55229.401942,793,70,616,55.626195,8.827238,125.203883,77.679697
1858,Jeans,Diesel,115950.979805,54506.868659,597,61,483,47.008545,10.217755,194.222747,80.904523
287,Accessories,Ray-Ban,78220.749924,47085.530239,656,68,496,60.195703,10.365854,119.238948,75.609756


In [18]:
evolucion = df.groupby(['year', 'mes']).agg({ 
    'revenue': 'sum',
    'margin': 'sum',
    'item_id': 'count',
    'is_returned': 'sum'
}).reset_index()

In [20]:
evolucion['Crecimiento_Ventas_%'] = evolucion['revenue'].pct_change() * 100

In [21]:
evolucion.to_csv("../data/processed/kpi_evolucion_temporal.csv", index=False)

In [22]:
evolucion.head()

,year,mes,revenue,margin,item_id,is_returned,Crecimiento_Ventas_%
0,2019,1,1965.080007,986.173184,32,3,NaN
1,2019,2,4015.890006,2103.029722,53,5,104.362672
2,2019,3,8088.340017,4205.258070,123,14,101.408405
3,2019,4,9104.450037,4653.924408,152,13,12.562652
4,2019,5,16128.430004,8458.945345,249,27,77.148866


B. Perfil del Cliente y Ubicación

En este bloque analizamos qué tipo de personas son las que realmente sostienen el negocio. Cruzamos los datos de ubicación con el género y la edad para entender mejor a nuestro público. Esto nos servirá para saber, por ejemplo, si en un país concreto nos compran más hombres jóvenes o mujeres de mediana edad.

In [27]:
bins = [0, 18, 30, 45, 60, 100]
labels = ['0-18', '19-30', '31-45', '46-60', '60+']
df['rango_edad'] = pd.cut(df['age'], bins=bins, labels=labels)

In [28]:
clientes = df.groupby(['country', 'rango_edad', 'gender', 'es_recurrente'], observed=True).agg({
    'revenue': 'sum',
    'user_id': 'nunique',
    'item_id': 'count',
    'is_returned': 'sum'
}).reset_index()

In [29]:
clientes.columns = ['Pais', 'Rango_Edad', 'Genero', 'Recurrente', 'Ventas_Totales', 'Clientes_Unicos', 'Numero_Pedidos', 'Devoluciones']

In [30]:
clientes['Gasto_por_Cliente'] = clientes['Ventas_Totales'] / clientes['Clientes_Unicos']
clientes['Tasa_Devolucion (%)'] = (clientes['Devoluciones'] / clientes['Numero_Pedidos']) * 100

In [31]:
clientes.to_csv("../data/processed/kpi_clientes.csv", index=False)

In [32]:
clientes.head()

,Pais,Rango_Edad,Genero,Recurrente,Ventas_Totales,Clientes_Unicos,Numero_Pedidos,Devoluciones,Gasto_por_Cliente,Tasa_Devolucion (%)
0,Australia,0-18,F,0,4206.430011,102,102,12,41.239510,11.764706
1,Australia,0-18,F,1,8840.560032,57,130,12,155.097544,9.230769
2,Australia,0-18,M,0,4875.040010,107,107,11,45.561122,10.280374
3,Australia,0-18,M,1,9959.280010,58,120,13,171.711724,10.833333
4,Australia,19-30,F,0,6988.200014,178,178,17,39.259551,9.550562


In [34]:
evolucion_clientes = df.groupby(['year', 'mes', 'es_recurrente']).agg({
    'user_id': 'nunique',
    'revenue': 'sum'
}).reset_index()

In [35]:
evolucion_clientes.to_csv("../data/processed/kpi_clientes_evolucion.csv", index=False)

In [36]:
evolucion_clientes.head()

,year,mes,es_recurrente,user_id,revenue
0,2019,1,0,12,340.079997
1,2019,1,1,15,1625.000010
2,2019,2,0,22,977.580002
3,2019,2,1,22,3038.310004
4,2019,3,0,52,1883.010007


C. Logística y Operaciones

En este bloque analizamos el ciclo de vida del pedido desde que el usuario hace clic en "Comprar" hasta que el paquete llega a su puerta.

Primero calculamos el tiempo de envio, que es basicamente la medida principal del bloque:

$Tiempo Envío = Fecha\_Entrega (delivered\_at) - Fecha\_Creación (created\_at)$

Luego calcularemos el tiempo de transporte de cada envío y el tiempo que el paquete está en el almacen preparandose

In [37]:
# Aseguramos formato fecha (errors='coerce' evita que el código rompa si hay celdas vacías)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['shipped_at'] = pd.to_datetime(df['shipped_at'], errors='coerce')
df['delivered_at'] = pd.to_datetime(df['delivered_at'], errors='coerce')

# KPI 1: Lead Time Total (Experiencia completa del cliente)
df['tiempo_envio_total'] = (df['delivered_at'] - df['created_at']).dt.days

# KPI 2: Lead Time de Envío (Días en tránsito)
df['tiempo_transito'] = (df['delivered_at'] - df['shipped_at']).dt.days

# KPI 3: Eficiencia de Almacén (Tiempo de preparación interna)
df['tiempo_preparacion'] = (df['shipped_at'] - df['created_at']).dt.days

In [38]:
kpi_logistica = df.groupby(['dist_center_name', 'country'], observed=True).agg({
    'item_id': 'count',
    'tiempo_envio_total': 'mean',
    'tiempo_transito': 'mean',
    'tiempo_preparacion': 'mean',
    'is_returned': 'sum'
}).reset_index()

In [39]:
kpi_logistica.columns = ['Centro_Distribucion', 'Pais', 'Total_Paquetes', 'Media_Envio_Total', 'Media_Transito', 'Media_Preparacion', 'Total_Devoluciones']

In [40]:
kpi_logistica['Tasa_Devolucion (%)'] = (kpi_logistica['Total_Devoluciones'] / kpi_logistica['Total_Paquetes']) * 100

In [41]:
kpi_logistica.to_csv("../data/processed/kpi_logistica.csv", index=False)

In [42]:
kpi_logistica.head()

,Centro_Distribucion,Pais,Total_Paquetes,Media_Envio_Total,Media_Transito,Media_Preparacion,Total_Devoluciones,Tasa_Devolucion (%)
0,Charleston SC,Australia,350,2.427419,1.919355,0.111111,35,10.000000
1,Charleston SC,Belgium,184,2.276596,1.872340,0.033333,15,8.152174
2,Charleston SC,Brasil,2440,2.634174,2.030963,0.078864,261,10.696721
3,Charleston SC,China,5845,2.585622,1.996006,0.057264,564,9.649273
4,Charleston SC,Colombia,4,4.500000,4.000000,0.666667,0,0.000000


In [43]:
kpi_logistica_evolucion = df.groupby(['year', 'mes']).agg({
    'tiempo_envio_total': 'mean',
    'tiempo_transito': 'mean',
    'is_returned': 'mean'
}).reset_index()

In [44]:
kpi_logistica_evolucion.to_csv("../data/processed/kpi_logistica_evolucion.csv", index=False)

In [45]:
kpi_logistica_evolucion.head()

,year,mes,tiempo_envio_total,tiempo_transito,is_returned
0,2019,1,3.000000,2.400000,0.093750
1,2019,2,3.000000,1.750000,0.094340
2,2019,3,2.416667,2.055556,0.113821
3,2019,4,3.285714,2.367347,0.085526
4,2019,5,2.666667,2.093750,0.108434


D. Estrategia de Producto y Calidad

En este bloque nos centraremos en la calidad del producto que vendemos fijandonos en la tasa de retorno que tiene. 
Además de ver la cantidad de ventas que tienen las categorias de productos por estación. Y así ver que se compra más en esas estaciones.

In [72]:
kpi_calidad = df.groupby(['brand', 'category']).agg({
    'item_id': 'count',
    'is_returned': 'sum',
    'revenue': 'sum',
    'margin': 'sum'
}).reset_index()

In [73]:
kpi_calidad.columns = ['Marca', 'Categoria', 'Total_Paquetes', 'Paquetes_Devueltos', 'Ventas_Totales', 'Margen_Bruto']

In [74]:
kpi_calidad['Tasa_Retorno'] = (kpi_calidad['Paquetes_Devueltos'] / kpi_calidad['Total_Paquetes']) * 100

In [75]:
kpi_calidad['Venta_Perdida'] = (kpi_calidad['Ventas_Totales'] / kpi_calidad['Total_Paquetes']) * kpi_calidad['Paquetes_Devueltos']

In [76]:
kpi_calidad = kpi_calidad.sort_values(by = 'Tasa_Retorno', ascending = False)

In [77]:
kpi_calidad.to_csv("../data/processed/kpi_calidad.csv", index=False)

In [78]:
kpi_calidad.head()

,Marca,Categoria,Total_Paquetes,Paquetes_Devueltos,Ventas_Totales,Margen_Bruto,Tasa_Retorno,Venta_Perdida
5171,Torrid,Blazers & Jackets,1,1,58.500000,37.908000,100.0,58.500000
3021,Kuhl,Shorts,1,1,64.949997,35.267848,100.0,64.949997
1984,Fox Racing,Socks,1,1,8.950000,3.946950,100.0,8.950000
808,Brave Soul,Jumpsuits & Rompers,1,1,13.200000,5.676000,100.0,13.200000
5794,eVogues Apparel,Jumpsuits & Rompers,1,1,29.990000,14.155280,100.0,29.990000


In [79]:
kpi_estacionalidad = df.groupby(['estacion', 'category'], observed=True).agg({
    'revenue': 'sum',
    'item_id': 'count',
    'is_returned': 'sum'
}).reset_index()

In [80]:
kpi_estacionalidad.columns = ['Estacion', 'Categoria', 'Ventas_Totales', 'Total_Paquetes', 'Paquetes_Devueltos']

In [81]:
kpi_estacionalidad['Tasa_Retorno_Estacional'] = (kpi_estacionalidad['Paquetes_Devueltos'] / kpi_estacionalidad['Total_Paquetes']) * 100

In [82]:
kpi_estacionalidad.to_csv("../data/processed/kpi_estacionalidad.csv", index=False)

In [83]:
kpi_estacionalidad.head()

,Estacion,Categoria,Ventas_Totales,Total_Paquetes,Paquetes_Devueltos,Tasa_Retorno_Estacional
0,Invierno,Accessories,118118.889835,2774,261,9.408796
1,Invierno,Active,136524.850072,2641,266,10.071942
2,Invierno,Blazers & Jackets,84582.980186,934,86,9.207709
3,Invierno,Clothing Sets,5275.860012,63,5,7.936508
4,Invierno,Dresses,138105.190290,1644,176,10.705596


In [86]:
kpi_calidad_evolucion = df.groupby(['year', 'mes']).agg({
    'is_returned': 'mean',
    'item_id': 'count'
}).reset_index()

In [87]:
kpi_calidad_evolucion.columns = ['Año', 'Mes', 'Media_Paquetes_Devueltos', 'Total_Paquetes']

In [88]:
kpi_calidad_evolucion.to_csv("../data/processed/kpi_calidad_evolucion.csv", index=False)

In [89]:
kpi_calidad_evolucion.head()

,Año,Mes,Media_Paquetes_Devueltos,Total_Paquetes
0,2019,1,0.093750,32
1,2019,2,0.094340,53
2,2019,3,0.113821,123
3,2019,4,0.085526,152
4,2019,5,0.108434,249
